# 6. ETL — Punjenje dimenzijskog modela

Ovaj notebook izvršava ETL (Extract, Transform, Load) proces:
1. Pražnjenje postojećih podataka u dimenzijskim tablicama
2. Punjenje dimenzija (`dim_projekt`, `dim_tehnicar`, `dim_prioritet_status`, `dim_vrijeme`)
3. Transformacija metrika (vrijeme rješavanja, workflow sati)
4. Punjenje tablice činjenica (`fact_support_tickets`)

**Preduvjeti:** Pokrenuti notebook `5_dimensional_ddl.ipynb` za kreiranje tablica.

In [8]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'fipu_srp_projekt')

if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD nije postavljen! Kreiraj .env datoteku.")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")
print(f"Spojeno na bazu: {DB_NAME} ({DB_HOST})")

Spojeno na bazu: fipu_srp_projekt (localhost)


## 6.1 Pražnjenje tablica

In [9]:
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE fact_support_tickets;"))
    conn.execute(text("TRUNCATE TABLE dim_vrijeme;"))
    conn.execute(text("TRUNCATE TABLE dim_projekt;"))
    conn.execute(text("TRUNCATE TABLE dim_tehnicar;"))
    conn.execute(text("TRUNCATE TABLE dim_prioritet_status;"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    conn.commit()
    print("Tablice ispražnjene.")

Tablice ispražnjene.


## 6.2 EXTRACT — Učitavanje izvornih podataka

In [10]:
df = pd.read_csv('Support_tickets_PROCESSED.csv')
df['issue_created'] = pd.to_datetime(df['issue_created'], format='ISO8601')
df['issue_resolution_date'] = pd.to_datetime(df['issue_resolution_date'], format='ISO8601')
print(f"Učitano {len(df)} redaka. Raspon: {df['issue_created'].min()} — {df['issue_created'].max()}")

Učitano 53353 redaka. Raspon: 2007-04-15 08:12:11+00:00 — 2023-03-14 14:00:32+00:00


## 6.3 LOAD — Punjenje dimenzija

In [11]:
# DIM_PROJEKT
dim_projekt = df[['issue_proj']].drop_duplicates().reset_index(drop=True)
dim_projekt.columns = ['naziv_projekta']
dim_projekt.to_sql('dim_projekt', engine, if_exists='append', index=False)
db_projekti = pd.read_sql("SELECT projekt_key, naziv_projekta FROM dim_projekt", engine)
print(f"dim_projekt: {len(dim_projekt)} redaka")

# DIM_TEHNICAR
reporters = df[['issue_reporter']].rename(columns={'issue_reporter': 'ime_prezime'})
assignees = df[['issue_assignee']].rename(columns={'issue_assignee': 'ime_prezime'})
dim_tehnicar = pd.concat([reporters, assignees]).drop_duplicates().dropna().reset_index(drop=True)
dim_tehnicar.to_sql('dim_tehnicar', engine, if_exists='append', index=False)
db_tehnicari = pd.read_sql("SELECT tehnicar_key, ime_prezime FROM dim_tehnicar", engine)
print(f"dim_tehnicar: {len(dim_tehnicar)} redaka")

# DIM_PRIORITET_STATUS
dim_ps = df[['issue_priority', 'issue_status']].drop_duplicates().reset_index(drop=True)
dim_ps.columns = ['razina_prioriteta', 'naziv_statusa']
dim_ps.to_sql('dim_prioritet_status', engine, if_exists='append', index=False)
db_ps = pd.read_sql("SELECT prioritet_status_key, razina_prioriteta, naziv_statusa FROM dim_prioritet_status", engine)
print(f"dim_prioritet_status: {len(dim_ps)} redaka")

# DIM_VRIJEME
dates = df['issue_created'].dt.date.unique()
dim_vrijeme = pd.DataFrame({'vrijeme_key': pd.to_datetime(dates)})
dim_vrijeme['dan'] = dim_vrijeme['vrijeme_key'].dt.day
dim_vrijeme['mjesec'] = dim_vrijeme['vrijeme_key'].dt.month
dim_vrijeme['godina'] = dim_vrijeme['vrijeme_key'].dt.year
dim_vrijeme['kvartal'] = dim_vrijeme['vrijeme_key'].dt.quarter
dim_vrijeme['dan_u_tjednu'] = dim_vrijeme['vrijeme_key'].dt.day_name()
dim_vrijeme.to_sql('dim_vrijeme', engine, if_exists='append', index=False)
print(f"dim_vrijeme: {len(dim_vrijeme)} redaka")

dim_projekt: 15 redaka
dim_tehnicar: 100 redaka
dim_prioritet_status: 54 redaka
dim_vrijeme: 4706 redaka


## 6.4 TRANSFORM — Izračun metrika

In [12]:
# Metrika 1: Ukupno vrijeme rješavanja (sati)
df['vrijeme_rjesavanja_sati'] = (
    (df['issue_resolution_date'] - df['issue_created']).dt.total_seconds() / 3600
)
df.loc[df['vrijeme_rjesavanja_sati'] < 0, 'vrijeme_rjesavanja_sati'] = 0
df['vrijeme_rjesavanja_sati'] = df['vrijeme_rjesavanja_sati'].round(2)

# Metrike 2-5: Vrijeme u workflow stanjima (sekunde -> sati)
df['sati_open'] = (df['wf_open'] / 3600).round(2)
df['sati_in_progress'] = (df['wf_in_progress'] / 3600).round(2)
df['sati_resolved'] = (df['wf_resolved'] / 3600).round(2)
df['sati_waiting'] = (df['wf_waiting'] / 3600).round(2)

print("Metrike izračunate:")
for col in ['vrijeme_rjesavanja_sati', 'sati_open', 'sati_in_progress', 'sati_resolved', 'sati_waiting']:
    print(f"  {col:30s} prosjek: {df[col].mean():10.1f} h, NULL: {df[col].isna().sum()}")

Metrike izračunate:
  vrijeme_rjesavanja_sati        prosjek:    18441.0 h, NULL: 698
  sati_open                      prosjek:    17819.5 h, NULL: 60
  sati_in_progress               prosjek:      266.8 h, NULL: 23918
  sati_resolved                  prosjek:       86.6 h, NULL: 30795
  sati_waiting                   prosjek:      481.5 h, NULL: 37466


## 6.5 TRANSFORM — Mapiranje stranih ključeva

In [13]:
fact = df.copy()

# Projekt (inner join)
fact = fact.merge(db_projekti, left_on='issue_proj', right_on='naziv_projekta', how='inner')
print(f"Nakon merge s dim_projekt: {len(fact)} redaka")

# Reporter (inner join — svaki ticket ima reportera)
fact = fact.merge(db_tehnicari, left_on='issue_reporter', right_on='ime_prezime', how='inner')
fact = fact.rename(columns={'tehnicar_key': 'reporter_key'})
print(f"Nakon merge s dim_tehnicar (reporter): {len(fact)} redaka")

# Assignee (LEFT join — 46% ticketa nema assignee)
fact = fact.merge(db_tehnicari, left_on='issue_assignee', right_on='ime_prezime',
                  how='left', suffixes=('', '_assignee'))
fact = fact.rename(columns={'tehnicar_key': 'assignee_key'})
print(f"Nakon merge s dim_tehnicar (assignee): {len(fact)} redaka (NULL: {fact['assignee_key'].isna().sum()})")

# Prioritet + Status (inner join)
fact = fact.merge(db_ps, left_on=['issue_priority', 'issue_status'],
                  right_on=['razina_prioriteta', 'naziv_statusa'], how='inner')
print(f"Nakon merge s dim_prioritet_status: {len(fact)} redaka")

Nakon merge s dim_projekt: 53353 redaka
Nakon merge s dim_tehnicar (reporter): 53353 redaka
Nakon merge s dim_tehnicar (assignee): 53353 redaka (NULL: 24771)
Nakon merge s dim_prioritet_status: 53353 redaka


## 6.6 LOAD — Punjenje fact tablice

In [14]:
# Priprema finalnog DataFrame-a prema DDL shemi
fact['vrijeme_key'] = fact['issue_created'].dt.date

fact_final = fact[[
    'id', 'projekt_key', 'reporter_key', 'assignee_key',
    'prioritet_status_key', 'vrijeme_key',
    'vrijeme_rjesavanja_sati', 'issue_comments_count',
    'sati_open', 'sati_in_progress', 'sati_resolved', 'sati_waiting'
]].copy()

fact_final = fact_final.rename(columns={
    'id': 'ticket_id',
    'issue_comments_count': 'broj_komentara'
})
fact_final['ticket_id'] = fact_final['ticket_id'].astype(int)
fact_final['assignee_key'] = fact_final['assignee_key'].astype('Int64')

# Bulk insert
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.commit()

fact_final.to_sql('fact_support_tickets', engine, if_exists='append', index=False, chunksize=5000)

with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    conn.commit()

print(f"ETL uspješno završen! Uneseno {len(fact_final)} redaka u fact_support_tickets.")

ETL uspješno završen! Uneseno 53353 redaka u fact_support_tickets.


## 6.7 Verifikacija

In [15]:
print("=" * 50)
print("VERIFIKACIJA ETL PROCESA")
print("=" * 50)

with engine.connect() as conn:
    for tbl in ['dim_projekt', 'dim_tehnicar', 'dim_prioritet_status', 'dim_vrijeme', 'fact_support_tickets']:
        cnt = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
        print(f"  {tbl:30s} {cnt:>6d} redaka")

print(f"\nOčekivano u fact tablici: {len(fact_final)}")

sample = pd.read_sql("""
    SELECT f.ticket_id, p.naziv_projekta, r.ime_prezime AS reporter,
           a.ime_prezime AS assignee, ps.razina_prioriteta, ps.naziv_statusa,
           f.vrijeme_key, f.vrijeme_rjesavanja_sati, f.broj_komentara,
           f.sati_open, f.sati_in_progress, f.sati_resolved, f.sati_waiting
    FROM fact_support_tickets f
    JOIN dim_projekt p ON f.projekt_key = p.projekt_key
    JOIN dim_tehnicar r ON f.reporter_key = r.tehnicar_key
    LEFT JOIN dim_tehnicar a ON f.assignee_key = a.tehnicar_key
    JOIN dim_prioritet_status ps ON f.prioritet_status_key = ps.prioritet_status_key
    LIMIT 10
""", engine)
print(f"\nUzorak:")
print(sample.to_string(index=False))

VERIFIKACIJA ETL PROCESA
  dim_projekt                        15 redaka
  dim_tehnicar                      100 redaka
  dim_prioritet_status               54 redaka
  dim_vrijeme                      4706 redaka
  fact_support_tickets            53353 redaka

Očekivano u fact tablici: 53353

Uzorak:
 ticket_id naziv_projekta    reporter    assignee razina_prioriteta naziv_statusa vrijeme_key  vrijeme_rjesavanja_sati  broj_komentara  sati_open  sati_in_progress  sati_resolved sati_waiting
     10701  Project Atlas Ivana Božić Josip Jurić           unknown        closed  2016-03-30                   510.07               2       0.02              0.22         509.83         None
     11718  Project Atlas  Ivan Babić        None            Medium          done  2015-08-12                 21483.78               0   21483.78               NaN            NaN         None
     11720  Project Atlas  Ivan Babić        None            Medium          done  2015-08-16                 21390.61    